# Perform FMQA with Covalent and Fixstars Amplify

This example uses Covalent and Fixstars Amplify to perform FMQA.

The code is a modified version of [Amplify Examples](https://github.com/fixstars/amplify-examples/blob/main/notebooks/ja/examples/fmqa_0_algebra.ipynb). Amplify Examples is open source software under the [MIT License](https://github.com/fixstars/amplify-examples/blob/main/LICENSE).

## Environment Setup

### Steps to create a virtual environment on the remote machine

To run this example, you need to setup an environment (or virtual environment) with the necessary packages installed on the remote machine.

Below are the steps to create a virtual environment on the remote machine named `amplify_env` with `venv`.

1. Login to the remote machine using your own account.
2. Verify that the Python version on the remote machine is the same as the one on your local machine, down to minor versions.
3. Create a virtual environment by running `python3 -m venv amplify_env`. The `amplify_env` directory is also created at this time.
4. Activate the virtual environment by running `source amplify_env/bin/activate`.
5. Install the necessary packages.
    * Please refer to [Dependent Packages](#dependencies) for the required packages.

<a id="dependencies"></a>
### Dependent Packages

This example requires the packages listed below. Please install the packages required to run this example in your local machine environment and in the virtual environment on the remote machine.  
Bolded packages are required for both the local machine and the virtual environment on the remote machine, and the rest are required only for the local machine.

This example uses covalent-pbspro-plugin.  
Please refer to the README of covalent-pbspro-plugin for information on how to install and use covalent-pbspro-plugin.

- **covalent**
- **numpy**
- **torch**
- **scikit-learn**
- amplify>=1.0.0
- matplotlib
- covalent-pbspro-plugin

In [ ]:
from __future__ import annotations
import covalent as ct

Create a PBSProExecutor object to execute tasks on the remote machine.

In [ ]:
remote_executor = ct.executor.PBSProExecutor(
    username="username",      # Enter your username on the remote machine.
    address="localhost",    # Enter the address of the remote machine.
    ssh_key_file="~/.ssh/id_rsa",  # Enter the path to your ssh key file.
    remote_workdir="$HOME/amplify_env",
    poll_freq=30,
    cleanup=True,
    embedded_qsub_args={
        "l": ["walltime=1:00:00"],
    },  # qsub options to be embedded in the script
    qsub_args={
    },  # qsub options to be given when it is run on the command line
    prerun_commands=[
        "source ~/amplify_env/bin/activate",
    ],
    postrun_commands=[],
    bashrc_path="~/.bashrc",
    log_stdout="stdout.log",
    log_stderr="stderr.log",
)

Prepare the necessary functions to run FMQA.

The following is an excerpt of code from [Amplify Examples](https://github.com/fixstars/amplify-examples/blob/main/notebooks/ja/examples/fmqa_0_algebra.ipynb) and modified to run using Covalent.

In [ ]:
import torch
import numpy as np

import torch.nn as nn

class TorchFM(nn.Module):
    def __init__(self, d: int, k: int) -> None:
        super().__init__()
        self.d = d
        self.v = torch.randn((d, k), requires_grad=True)
        self.w = torch.randn((d,), requires_grad=True)
        self.w0 = torch.randn((), requires_grad=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out_linear = torch.matmul(x, self.w) + self.w0

        out_1 = torch.matmul(x, self.v).pow(2).sum(1)
        out_2 = torch.matmul(x.pow(2), self.v.pow(2)).sum(1)
        out_quadratic = 0.5 * (out_1 - out_2)

        out = out_linear + out_quadratic
        return out

    def get_parameters(self) -> tuple[np.ndarray, np.ndarray, float]:
        np_v = self.v.detach().numpy().copy()
        np_w = self.w.detach().numpy().copy()
        np_w0 = self.w0.detach().numpy().copy()
        return np_v, np_w, float(np_w0)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

import copy

# This task is executed on the remote machine.
# A function that machine learns FM from I/O data
@ct.electron(executor=remote_executor)
def train(
    X: np.ndarray,
    y: np.ndarray,
    model: TorchFM,
) -> torch.nn.Module:
    epochs = 2000
    optimizer = torch.optim.AdamW([model.v, model.w, model.w0], lr=0.1)
    loss_func = nn.MSELoss()

    x_tensor, y_tensor = (
        torch.from_numpy(X).float(),
        torch.from_numpy(y).float(),
    )
    dataset = TensorDataset(x_tensor, y_tensor)
    train_set, valid_set = random_split(dataset, [0.8, 0.2])
    train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
    valid_loader = DataLoader(valid_set, batch_size=8, shuffle=True)

    min_loss = 1e18
    best_state = model.state_dict()

    for _ in range(epochs):
        for x_train, y_train in train_loader:
            optimizer.zero_grad()
            pred_y = model(x_train)
            loss = loss_func(pred_y, y_train)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            loss = 0
            for x_valid, y_valid in valid_loader:
                out_valid = model(x_valid)
                loss += loss_func(out_valid, y_valid)
            if loss < min_loss:
                # 損失関数の値が更新されたらパラメータを保存
                best_state = copy.deepcopy(model.state_dict())
                min_loss = loss

    model.load_state_dict(best_state)
    return model

In [ ]:
from collections.abc import Callable

# This task is executed on the remote machine.
# A function that evaluates the objective function for input values and creates N0 input-output pairs (initial training data).
@ct.electron(executor=remote_executor)
def init_training_data(D: int, N0: int, blackbox_func: Callable) -> tuple[np.ndarray, np.ndarray]:
    assert N0 < 2**D

    # Generate N0 input values using random numbers.
    rng = np.random.default_rng()
    x = rng.choice(np.array([0, 1]), size=(N0, D))

    # Exclude duplicate input values from the input values.
    # And add the excluded input values using random numbers.
    x = np.unique(x, axis=0)
    while x.shape[0] != N0:
        x = np.vstack((x, np.random.randint(0, 2, size=(N0 - x.shape[0], D))))
        x = np.unique(x, axis=0)
    y = np.zeros(N0)

    # Get the output value corresponding to the N0 input values by evaluating the objective function.
    for i in range(N0):
        y[i] = blackbox_func(x[i])

    return x, y

Note that the code below is implemented with Amplify SDK v1 and is not guaranteed to work with Amplify SDK v0.  
Please see following link for detail.
https://amplify.fixstars.com/en/docs/amplify/v1/migration.html

In [ ]:
import amplify

# This task is executed on the local machine.
# A function that performs a solution using a Ising machine.
@ct.electron
def anneal(torch_model: TorchFM, D: int, k: int) -> np.ndarray:
    client = amplify.FixstarsClient()
    client.parameters.timeout = 1000
    client.token = "xxxxxxxxxxxxxxx"  # Enter your token of Amplify AE.

    gen = amplify.VariableGenerator()
    x = gen.array("Binary", torch_model.d)

    v, w, w0 = torch_model.get_parameters()

    out_linear = w0 + (x * w).sum()
    out_1 = ((x[:, np.newaxis] * v).sum(axis=0) ** 2).sum()  # type: ignore
    out_2 = ((x[:, np.newaxis] * v) ** 2).sum()
    objective: amplify.Poly = out_linear + (out_1 - out_2) / 2

    amplify_model = amplify.Model(objective)
    result = amplify.solve(amplify_model, client)  # Solve QUBO using Ising machine.
    if len(result.solutions) == 0:
        raise RuntimeError("No solution was found.")

    return x.evaluate(result.best.values).astype(int)


import matplotlib.pyplot as plt

# A function that plots the history of objective function evaluation values for the initial training data and the i-th FMQA cycle.
def plot_history(y: np.ndarray, N: int, N0: int) -> plt.Figure:
    assert y is not None
    fig = plt.figure(figsize=(6, 4))
    plt.plot(
        [i for i in range(N0)],
        y[: N0],
        marker="o",
        linestyle="-",
        color="b",
    )  # Objective function evaluation value for the initial training data (random process)

    plt.plot(
        [i for i in range(N0, N0 + N)],
        y[N0 :],
        marker="o",
        linestyle="-",
        color="r",
    )  # Objective function evaluation value for the i-th FMQA cycle.
    plt.xlabel("number of iterations", fontsize=18)
    plt.ylabel("f(x)", fontsize=18)
    plt.tick_params(labelsize=18)
    plt.show()
    return fig

In [ ]:
# Create a d-dimensional symmetric matrix with zero mean of the components.

def make_blackbox_func(D: int) -> Callable:
    rng = np.random.default_rng()
    Q = rng.random((D, D))
    Q = (Q + Q.T) / 2
    Q = Q - np.mean(Q)
    def blackbox(x):
        return x @ Q @ x
    return blackbox


In [ ]:
# This task is executed on the remote machine.
@ct.electron(executor=remote_executor)
def append_new_data(
    x: np.ndarray,
    y: np.ndarray,
    x_hat: np.ndarray,
    D: int,
    blackbox_func: Callable,
) -> tuple[np.ndarray, np.ndarray]:
    # If the same input value already exists in the training data as x_hat, the surrounding values are used as x_hat.
    rng = np.random.default_rng()
    while (x_hat == x).all(axis=1).any():
        flip_idx = rng.choice(np.arange(D))
        x_hat[flip_idx] = 1 - x_hat[flip_idx]

    y_hat = blackbox_func(x_hat)

    x = np.vstack((x, x_hat))
    y = np.append(y, y_hat)
    return x, y

Combine the above 4 `electron` to create a `lattice` workflow.

In [ ]:
@ct.lattice
def workflow(D: int, N: int, N0: int, k: int) -> tuple[np.ndarray, np.ndarray]:
    blackbox = make_blackbox_func(D)
    x, y = init_training_data(D, N0, blackbox)

    for i in range(N):
        model = TorchFM(D, k)
        model = train(x, y, model)
        x_hat = anneal(model, D, k)

        x, y = append_new_data(x, y, x_hat, D, blackbox)

    return x, y


In [ ]:
D = 100 # Input dimension (problem size)
N = 10  # Number of times the function can be evaluated
N0 = 60 # Number of initial training data samples
k = 10  # Dimension of the vector in FM (hyper parameter)

Next, start the covalent server with the following command

```console
covalent start
````

With the default configuration, you can view the Covalent GUI by accessing http://localhost:48008/ from your browser.

Finally, execute the workflow. First, dispatch the workflow.

In [ ]:
dispatch_id = ct.dispatch(workflow)(D, N, N0, k)

In [ ]:
print(dispatch_id)

In [ ]:
result = ct.get_result(dispatch_id, wait=True)
print(result)

Plot the history of the objective function evaluation values.

In [ ]:
x, y = result.result
min_idx = np.argmin(y)
print(f"best x = {x[min_idx]}")
print(f"best y = {y[min_idx]}")

fig = plot_history(y, N, N0)